# Module 8: Information Theory

Information Theory, originally developed by Claude Shannon to study telecommunications, forms the backbone of loss functions (like cross-entropy), regularizers, and generative models (like VAEs, GANs, and Diffusion models) in modern machine learning.

## Contents
1. Shannon Entropy & Information Content
2. Joint and Conditional Entropy
3. Mutual Information
4. Cross-Entropy and Kullback-Leibler (KL) Divergence
5. The Data Processing Inequality & Channel Capacity
6. AI Use Cases: Loss Functions & Generative Models

## 1. Shannon Entropy & Information Content

The **Information Content** (or Self-Information) of an event $x$ with probability $P(x)$ is:
$$I(x) = -\log_2 P(x)$$
This measures the "surprise" of the event. If an event is certain ($P(x)=1$), its information content is 0. If it is rare ($P(x) \to 0$), its information content is extremely high.

The **Shannon Entropy** $H(X)$ of a discrete random variable $X$ is the expected value of its self-information:
$$H(X) = \mathbb{E}[I(X)] = -\sum_{x \in \mathcal{X}} P(x) \log_2 P(x)$$
Entropy measures the average uncertainty or randomness in a probability distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Entropy of a Bernoulli trial (coin flip with bias p)
p = np.linspace(0.001, 0.999, 100)
entropy = -(p * np.log2(p) + (1 - p) * np.log2(1 - p))

plt.figure(figsize=(8, 5))
plt.plot(p, entropy, 'g-')
plt.xlabel("Probability of heads (p)")
plt.ylabel("Entropy (bits)")
plt.title("Entropy of a Bernoulli Trial (Binary Entropy Function)")
plt.grid(True)
plt.show()

## 2. Joint and Conditional Entropy

For two discrete random variables $X$ and $Y$:
- **Joint Entropy**: The total uncertainty associated with the pair $(X, Y)$:
  $$H(X, Y) = -\sum_{x \in \mathcal{X}} \sum_{y \in \mathcal{Y}} P(x, y) \log_2 P(x, y)$$
- **Conditional Entropy**: The uncertainty of $Y$ given that $X$ is known:
  $$H(Y|X) = -\sum_{x \in \mathcal{X}} \sum_{y \in \mathcal{Y}} P(x, y) \log_2 P(y|x)$$

**Chain Rule of Entropy**:
$$H(X, Y) = H(X) + H(Y|X)$$
Since conditioning never increases entropy on average, we have $H(Y|X) \le H(Y)$ (with equality iff $X$ and $Y$ are independent).

## 3. Mutual Information

**Mutual Information** $I(X; Y)$ measures the amount of information obtained about one random variable through observing the other:
$$I(X; Y) = H(X) - H(X|Y) = H(Y) - H(Y|X) = H(X) + H(Y) - H(X, Y)$$

Formally, it is defined as the KL divergence between the joint distribution and the product of their marginal distributions:
$$I(X; Y) = D_{KL}(P(X,Y) \parallel P(X)P(Y)) = \sum_{x, y} P(x, y) \log \frac{P(x, y)}{P(x)P(y)}$$

Unlike correlation, which only measures linear relationships, mutual information measures any dependency (linear or non-linear) between variables.

In [ ]:
from sklearn.metrics import mutual_info_score

# Generate dependent variables
X = np.random.randint(0, 2, 1000)
Y = X.copy()  # perfect correlation
print("Mutual Information (perfect dependency):", mutual_info_score(X, Y))

Z = np.random.randint(0, 2, 1000)  # independent
print("Mutual Information (independent):", mutual_info_score(X, Z))

## 4. Cross-Entropy and Kullback-Leibler (KL) Divergence

### Kullback-Leibler (KL) Divergence
Also called relative entropy, $D_{KL}(P \parallel Q)$ measures the inefficiency of coding messages from a true distribution $P$ using an estimated/model distribution $Q$:
$$D_{KL}(P \parallel Q) = \sum_{x \in \mathcal{X}} P(x) \log \frac{P(x)}{Q(x)}$$

Properties:
- $D_{KL}(P \parallel Q) \ge 0$ (Gibbs' Inequality, always non-negative)
- $D_{KL}(P \parallel Q) = 0$ iff $P = Q$
- Non-symmetric: $D_{KL}(P \parallel Q) \ne D_{KL}(Q \parallel P)$

### Cross-Entropy
The total bits required to code samples from $P$ using $Q$:
$$H(P, Q) = H(P) + D_{KL}(P \parallel Q) = -\sum_{x \in \mathcal{X}} P(x) \log Q(x)$$

In [ ]:
# Compute KL Divergence between two distributions
def kl_divergence(p, q):
    p = np.array(p, dtype=float)
    q = np.array(q, dtype=float)
    return np.sum(p * np.log(p / q))

p = [0.5, 0.5]
q = [0.9, 0.1]
print("KL(P || Q):", kl_divergence(p, q))
print("KL(Q || P):", kl_divergence(q, p))

## 5. The Data Processing Inequality & Channel Capacity

### Data Processing Inequality
For any Markov chain $X \to Y \to Z$, we cannot increase information about $X$ by processing $Y$:
$$I(X; Y) \ge I(X; Z)$$
This means no post-processing of data $Y$ can create information about $X$ that wasn't already present in $Y$.

### Channel Capacity
The maximum rate at which information can be transmitted reliably over a communication channel with transition probabilities $P(y|x)$:
$$C = \max_{P(x)} I(X; Y)$$

## 6. AI Use Cases: Loss Functions & Generative Models

- **Cross-Entropy Loss**: Standard training objective for classification models.
- **Variational Autoencoders (VAEs)**: Uses a KL divergence term to force the latent space to approximate a simple Gaussian prior $Q(z) = \mathcal{N}(0, I)$:
  $$\mathcal{L}_{VAE} = \mathbb{E}_{q_\phi(z|x)}[\log p_\theta(x|z)] - D_{KL}(q_\phi(z|x) \parallel p(z))$$
- **Information Bottleneck**: Used in deep representation learning to find a representation $Z$ of inputs $X$ that is highly predictive of outputs $Y$ but retains minimal excess information from $X$:
  $$\min I(X; Z) - \beta I(Y; Z)$$